# Number of threatened vertebrates in global rangelands

This notebook contains the steps to calculate the number of threatened terrestrial vertebrate species in rangelands around the globe, following the example given in the [Rangelands Atlas](https://www.rangelandsdata.org/atlas/maps/numbers-threatened-vertebrates-rangelands-globally).

## Methodology

First, we need to download the data from [IUCN](https://www.iucnredlist.org/resources/spatial-data-download). This requires credentials and also answering some questions to request the data. At this point we have downloaded data for Amphibians, Mammals and Reptiles. Data for Birds is still to be confirmed.


## Setup

### Library import


In [ ]:
import geopandas as gpd
import pandas as pd
import os
import logging
import subprocess
from pathlib import Path

In [ ]:
# Create a logger
logger = logging.getLogger(__name__)

# Set the log level to INFO
logger.setLevel(logging.INFO)

### Utils

In [ ]:
def read_and_preprocess(file_path, columns, terrestrial_flag='true'):
    data = gpd.read_file(file_path, columns=columns)
    data = data[data['terrestria'] == terrestrial_flag]
    return data[[col for col in columns if col != 'terrestria']]

def filter_categories(dataframes):
    return [df[df['category'].isin(['VU', 'EN', 'CR'])] for df in dataframes]

def create_mbtiles(
    source_path: Path,
    output_path: Path,
    layer_name: str,
    max_zoom: int,
    opts="--read-parallel --no-tile-compression -s EPSG:4326 -B4",
):
    """
    Use tippecanoe to create pbf tiles at dest_path from source_path (geojson).
    layer_name is used for the name of the layer in the MBTILE.
    Regex file path (/*.geojson) is supported for source_path.
    This function replaces the previous two functions (create_mbtiles & mbtile_to_pbf).

    More info: https://github.com/mapbox/tippecanoe#options

    Args:
        source_path (Path): path to source geojson
        output_path (Path): path to output .mbtiles
        layer_name (str): name of layer in the MBTILE
        max_zoom (int): max zoom level
        opts (str): options for tippecanoe

    Returns:
        (int): 0 if the file was created successfully, 1 if the file creation failed.
    """
    try:
        opts += f" -z{max_zoom}"
        cmd = f"tippecanoe -o {output_path} -l {layer_name} {opts} {source_path}"
        logger.info(f"Processing: {cmd}")
        r = subprocess.call(cmd, shell=True)
        if r == 0:
            logger.info("Task created")
        return r

    except Exception as e:
        logger.error(e)
        return 1


<a id='section_1'></a>
## Read and prepare data


In [ ]:
path_in = "/Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/raw/"
path_out = "/Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/processed/"

**Species**

In [ ]:
# Paths for species data
mammal_paths = [
    path_in + "MAMMALS_2025/MAMMALS_PART1.shp",
    path_in + "MAMMALS_2025/MAMMALS_PART2.shp"
]
amphibian_paths = [
    path_in + "AMPHIBIANS_2025/AMPHIBIANS_PART1.shp",
    path_in + "AMPHIBIANS_2025/AMPHIBIANS_PART2.shp"
]
reptile_paths = [
    path_in + "REPTILES_2025/REPTILES_PART1.shp",
    path_in + "REPTILES_2025/REPTILES_PART2.shp"
]


In [ ]:
# Columns to keep
species_columns = ['id_no', 'category', 'terrestria', 'geometry']

# Read and preprocess each group with the correct terrestrial column
mammals1 = read_and_preprocess(mammal_paths[0], species_columns)
mammals2 = read_and_preprocess(mammal_paths[1], species_columns)
amphibians1 = read_and_preprocess(amphibian_paths[0], species_columns)
amphibians2 = read_and_preprocess(amphibian_paths[1], species_columns)
reptiles1 = read_and_preprocess(reptile_paths[0], species_columns)
reptiles2 = read_and_preprocess(reptile_paths[1], species_columns)


In [ ]:
# Combine amphibian and reptile data
amphibians = pd.concat([amphibians1, amphibians2], ignore_index=True)
mammals = pd.concat([mammals1, mammals2], ignore_index=True)
reptiles = pd.concat([reptiles1, reptiles2], ignore_index=True)

# Keep only species that are vulnerable, endangered or critically endangered
mammals, amphibians, reptiles = filter_categories([mammals, amphibians, reptiles])

# Combine all species data
all_species = pd.concat([mammals, amphibians, reptiles], ignore_index=True)

**Ecoregions**

In [ ]:
# Path and columns for ecoregions
ecoregion_path = "../data/processed/ecoregions_2017.shp"
ecoregions_columns = ['geometry', 'ECO_NAME']


# Read ecoregions data
ecoregions = gpd.read_file(ecoregion_path)[ecoregions_columns]

### Calculate number of species in each ecoregion

In [ ]:
# Spatial join to find which species are in which ecoregion
species_in_ecoregions = gpd.sjoin(all_species, ecoregions, how='left', op='intersects')

In [ ]:
species_in_ecoregions

In [ ]:
# Group by ecoregion and biome, then count unique species and threat categories
species_count_ecoregion = species_in_ecoregions.groupby(['ECO_NAME']).agg(
    num_species=('id_no', 'nunique')).reset_index()

species_count_ecoregion

In [ ]:
# Add column num_species to ecoregions
ecoregions_species = ecoregions.merge(species_count_ecoregion, on='ECO_NAME', how='left')

# Convert NaN to 0
ecoregions_species['num_species'] = ecoregions_species['num_species'].fillna(0)
ecoregions_species

In [ ]:
# Create bins and labels for the number of species
bins = [0, 10, 25, 50, 75, 100, 300]
labels = ['0-10', '10-25', '25-50', '50-75', '75-100', '> 100']

ecoregions_species['bins'] = pd.cut(ecoregions_species['num_species'], bins=bins, labels=labels, right=False).astype(str)

ecoregions_species


In [ ]:
# Save as geojson
ecoregions_species.to_file("../data/processed/ecoregions_species_2025.geojson", driver='GeoJSON')

In [ ]:
create_mbtiles(
    os.path.join(path_out, "ecoregions_species_2025.geojson"),
    os.path.join(path_out, "ecoregions_species_2025.mbtiles"),
    "Threatened",
    12,
    "--force --read-parallel -zg -Z2 --drop-densest-as-needed --extend-zooms-if-still-dropping",
)
